# Navigation and Position QC

Ensure you have run the Prepare_XYZ notebook first so that the data are prepared for review.

___

Import the required modules, and set the path to the geowhizz files.

In [ ]:
from pathlib import Path
import galileoQC as qc

In [ ]:
EastVicHDF_file = Path(r'./EastVicData/EastVic.hdf5')
EastVicTestHDF_file = Path(r'./EastVicData/EastVicTest.hdf5')
EastVicRepHDF_file = Path(r'./EastVicData/EastVicRepeats.hdf5')
EastVicHDF_plan = Path(r'./EastVicData/EastVicPlan.hdf5')

In [ ]:
if not (EastVicHDF_file.exists() and EastVicTestHDF_file.exists() and EastVicRepHDF_file.exists() and EastVicHDF_plan.exists()):
    %run ./Prepare_EastVicData.ipynb

___

**GNSS**

Accurate knowledge of the sensor position is important. In gravimetry, it is critical for calculating the kinematic accelerations that must be removed from the gravimeter measurements. In gradiometry the accuracy requirement is not quite as demanding but it is still important since it limits the accuracy of the terrain corrections.

The accuracy of GNSS (Global Navigation Satellite System) positions is usually better when the number of visible satellites, `NSATS`, is large and the position, vertical and horizontal dilutions of position (PDOP, `VDOP` and `HDOP` respectively) are small. Contracts will often stipulate the minimum number of satellites, and maximum values for PDOP, VDOP and HDOP. The requirements tend to be a little more demanding for gravimetry than for gradiometry.

Typical values might be:

    airborne gravimetry - a minimum of 6 GNSS satellites visible, and PDOP, HDOP and VDOP to not exceed 4;
    airborne gradi0metry - a minimum of 5 GNSS satellites visible, and PDOP, HDOP and VDOP to not exceed 4.   

The values of these variables (number of satellites, HDOP, VDOP and PDOP) must be recorded as channels in the data, for each data point. The values in these channels are checked:

In [ ]:
qc.checkGNSS(EastVicHDF_file, 'NSATS', 'PDOP', 'VDOP', 'HDOP', nsats_min=5, max_pdop=4, 
                  max_hdop=4, max_vdop=4)

___

**Horizontal Position**

Typically surveys are planned so that the aircraft traverses a series of parallel lines, collecting data along each. The line spacing is an important determinant of the spatial resolution of the survey. Furthermore, the data are usually gridded at a cell spacing approximately one-fifth of the planned line spacing.

If the actual flight path deviates significantly from the planned path, then there will be areas of the survey where the spatial resolution of the data may be poorer than desired, and there will be areas where the gridding must interpolate over more grid cells between measurements than planned. Interpolation errors will be accordingly greater in those areas as a consequence.

The aircraft position is allowed to deviate horizontally from the planned survey line by more than `allowance` metres for a distance no greater than `maxDistance` metres. Options exist to limit the distance to that traversed in a given `maxDuration` number of seconds, or a given `maxCounter` number of fiducials.

Ideally the `allowance` would be much smaller than the intended grid cell width, and the `maxDistance` no greater than about one cell width. For gravimetry and gravity gradiometry these ideal restrictions can lead, with typical survey aircraft, to situations in which the instrumentation experiences significant accelerations as the aircraft path is corrected. These large aircraft accelerations can result in increased sensor error so it is better to allow a larger `maxDistance` and optimise for gentle corrections to the flight path. A 1 km distance is appropriate.

The check in this example requires that all flown lines are within $40\,m$ of the planned line, with any exceedances persisting for less than $1000\,m$. The check is repeated for just line `2345.510` to demonstrate the plotting that occurs should a line fail the check.

In [ ]:
qc.checkXYPlan(EastVicHDF_plan, EastVicHDF_file, allowance=40.0, maxCounter=0,
               maxDistance=1000.0, plot_flag=False)

In [ ]:
%matplotlib widget
qc.checkXYPlan(EastVicHDF_plan, EastVicHDF_file, allowance=300.0, maxCounter=0,
               maxDistance=1000.0, plot_flag=True)

___

**Vertical Position**

The vertical positions of the measurements are also constrained. The spatial resolution of the data is affected by the ground clearance, the sample spacing, and the filtering applied to the data. It is desirable to fly sufficiently low that the spatial resolution is not limited by survey height. The climb rate of the aircraft, as well as the presence of any obstacles limits this in practice.

Specifications might require the position to be close to a given height above the ground, or some constant altitude relative to a datum (geoid or ellipsoidal altitude) but often the requirement is that the aircraft remain within a given distance of a preplanned survey height, often called the drape.

Here we check that the aircraft remains within $20\,m$ of the planned height, with no exceedances persisting for more than $1000\,m$.

In [ ]:
qc.checkVertPlan(EastVicHDF_plan, EastVicHDF_file, allowance=20.0, maxCounter=0,
                 maxDistance=1000.0, plot_flag=True)

___

**Segment Lengths**

For operational reasons, a single planned survey line might be flown as a series of line segments, or as one segment. The length of all individually flown segments must exceed some minimum length. There are several reasons for setting a minimum. Where data are to be tie-line levelled, the segment must cross several control lines and so must be longer than two or three control line separations. For analysis of the spectral content or the statistics of the data, the segment length should be much longer than the spatial resolution of the data. And, for gradiometer surveys, the segment length should be large enough to allow for the noise reduction using regression against dynamic variables.

Typically the minimum length might around $10\,km$ for gradiometry and around $40\,km$ for gravimetry. This gravimetry analysis sets the minimum at $38\,km$.

In [ ]:
qc.checkLineLengths(EastVicHDF_file, min_len=38.0)

___

**Line Overlaps**

When a survey line is flown as more than one segment, the segments must overlap by some minimum amount in order to avoid any end-of-line effects (including those due to filtering).

In [ ]:
qc.checkOverlaps(EastVicHDF_file, min_overlap=10., plot_flag=False)

___

**Heading**

Occasionally contracts will set a specification on aircraft heading. Some care is required here because there is some risk of confusion about definitions. The track is the angle made between North and the line of motion of the aircraft (imagined as a track projected onto flat ground). The angle between North and the line defined by the nose-to-tail of the aircraft is the heading. Any cross wind will mean that the heading is different to the track. Nevertheless sometimes heading is reported as track, or vice versa.

Another potential cause of confusion is the distinction between magnetic North and true North. Here we mean true North.

And sometimes heading is called bearing.

The checkHeading function checks the given `heading` channel against the given allowed angles (in degrees) and reports any differences greater than some allowed difference. If the `heading` channel contains track data, then that is what is checked. If the name of the `heading` channel is not supplied, then `checkHeading` calculates the **track** angle from the position data and checks against that.

**Be warned**

In [ ]:
qc.checkHeading(EastVicHDF_file, [232.0, 52], plot_flag=True)

___

**Ground Speed**

The data are measured at constant sample rate in time but we are interested in the location of the data in space and the data will be at a constant spacing on the ground only if the aircraft ground speed is constant. Tail winds and head winds make this difficult and some variation is inevitable.

When the aircraft flies slower (into a head wind) the data have better spatial resolution, but a tail wind causes a faster ground speed and consequently poorer spatial resolution. In most conditions, a survey aircraft can fly within $15\%$ - $20\%$ of nominal, although nominal can vary significantly with aircraft.

This example requires the data to be within $15\%$ of $60\,ms^{-2}$, with exceedances to persist for no further than $1000\,m$.

In [ ]:
qc.checkSpeeds(EastVicHDF_file, nominalSpeed=55., allowance=0.15, maxDistance=1000.0, plot_flag=True)